# DeepSleepNet-Lite: Automatic Sleep Stage Classification

**CSEP 590A Deep Learning — Group Project**

This notebook runs the full DeepSleepNet-Lite pipeline on Google Colab:
1. Install dependencies and clone the repo
2. Download and preprocess Sleep-EDF data from PhysioNet
3. Train the model (single fold or full k-fold cross-validation)
4. Run predictions and evaluate performance
5. Visualize results (confusion matrix, training curves, class distribution)

**Runtime:** Go to `Runtime > Change runtime type` and select **GPU** (T4) before running.

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

# Install additional dependencies (Colab already has TensorFlow pre-installed)
!pip install -q tensorflow-probability mne scikit-learn scipy matplotlib seaborn

# Verify TensorFlow
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
import os

# Clone the repo (change this URL to your team's fork)
REPO_URL = "https://github.com/manishdas/deepsleepnet-lite.git"
REPO_DIR = "/content/deepsleepnet-lite"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull
    print(f"Repo already exists at {REPO_DIR}, pulled latest changes.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

## 2. Data Download and Preprocessing

The **Sleep-EDF** dataset (v1, 2013) contains polysomnography (PSG) recordings from 20 healthy subjects.
Each recording includes EEG channels (Fpz-Cz and Pz-Oz) and expert-annotated sleep stage labels.

- **5 sleep stages:** Wake (W), N1, N2, N3, REM
- **30-second epochs** at 100 Hz sampling rate (3000 samples per epoch)
- **20-fold leave-one-subject-out** cross-validation

Run Cell 2a below to download the data from Google Drive (~1 min).

In [ ]:
# ===== Cell 2a: Download data from Google Drive (FAST) =====
import os, glob

DATA_DIR = "data/eeg_FpzCz_PzOz_v1"

!pip install gdown -q

if not os.path.exists('data'):
    !gdown 1wDu9tl6_P250k522tQC9LUUVh7ocG1_x -O data.zip
    !unzip -q data.zip
    print('Data extracted.')
else:
    print('Data directory already exists.')

# Check if DATA_DIR already has the correct EEG files (SC4*E0.npz)
eeg_files = sorted(glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz')))
if not eeg_files:
    # Remove any wrong symlink
    if os.path.islink(DATA_DIR):
        os.unlink(DATA_DIR)
    # Search for the actual EEG data files anywhere under data/
    sc_files = sorted(glob.glob('data/**/SC4*E0.npz', recursive=True))
    if sc_files:
        actual_dir = os.path.dirname(sc_files[0])
        print(f"Found {len(sc_files)} EEG files in {actual_dir}")
        os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
        os.symlink(os.path.abspath(actual_dir), DATA_DIR)
        print(f"Linked {DATA_DIR} -> {actual_dir}")
        eeg_files = sorted(glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz')))

if eeg_files:
    print(f"Found {len(eeg_files)} processed EEG files -- ready to train!")
else:
    print("ERROR: No SC4*E0.npz files found. Check zip contents:")
    !find data/ -name "SC4*" | head -10

# Ensure data_split_v1.npz exists
split_path = os.path.join(os.path.dirname(DATA_DIR), 'data_split_v1.npz')
if not os.path.exists(split_path):
    import sys; sys.path.insert(0, '.')
    from prepare_physionet import create_data_splits
    create_data_splits(20, 20, split_path)

# Verify
npz_files = sorted(glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz')))
print(f"\nData ready: {len(npz_files)} files in {DATA_DIR}")

In [ ]:
# ===== Cell 2b (ALTERNATIVE): Load from Google Drive =====
# Use this if a teammate already processed the data and saved it to Drive.
# Skip this cell if you ran Cell 2a above.

# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_DATA = '/content/drive/MyDrive/SleepStageNet/data'
# DATA_DIR = 'data/eeg_FpzCz_PzOz_v1'
#
# import shutil, os
# os.makedirs('data', exist_ok=True)
# shutil.copytree(os.path.join(DRIVE_DATA, 'eeg_FpzCz_PzOz_v1'), DATA_DIR, dirs_exist_ok=True)
# shutil.copy2(os.path.join(DRIVE_DATA, 'data_split_v1.npz'), 'data/data_split_v1.npz')
# print(f"Loaded {len(os.listdir(DATA_DIR))} files from Drive")

In [ ]:
# Sanity check: verify data files and inspect one recording
import numpy as np
import matplotlib.pyplot as plt

# List all processed files
data_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.npz')])
print(f"Number of .npz files: {len(data_files)}")
print(f"Files: {data_files}\n")

# Load and inspect one file
sample_file = os.path.join(DATA_DIR, data_files[0])
with np.load(sample_file) as f:
    x = f['x']
    y = f['y']
    fs = f['fs']
print(f"Sample file: {data_files[0]}")
print(f"  x shape: {x.shape} (n_epochs, samples_per_epoch, channels)")
print(f"  y shape: {y.shape} (n_epochs,)")
print(f"  fs: {fs} Hz")

# Plot a sample 30-second epoch
stage_names = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}
sample_idx = np.where(y == 2)[0][0]  # pick an N2 epoch
t = np.arange(3000) / 100.0  # time in seconds

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, x[sample_idx, :, 0] * 1e6, linewidth=0.5)  # convert to microvolts
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (uV)')
ax.set_title(f'Sample EEG Epoch (Fpz-Cz) — Stage: {stage_names[y[sample_idx]]}')
plt.tight_layout()
plt.show()

# Overall class distribution
all_labels = []
for f_name in data_files:
    with np.load(os.path.join(DATA_DIR, f_name)) as f:
        all_labels.append(f['y'])
all_labels = np.concatenate(all_labels)

fig, ax = plt.subplots(figsize=(8, 4))
counts = [np.sum(all_labels == i) for i in range(5)]
bars = ax.bar(['Wake', 'N1', 'N2', 'N3', 'REM'], counts,
              color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#9b59b6'])
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count}\n({100*count/len(all_labels):.1f}%)',
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Number of Epochs')
ax.set_title(f'Sleep Stage Distribution — Sleep-EDF v1 ({len(all_labels)} total epochs)')
plt.tight_layout()
plt.savefig('figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved figures/class_distribution.png")

## 3. Model Training

**DeepSleepNet-Lite** uses two parallel CNN paths with different filter sizes to capture both
fine-grained and coarse temporal features from 90-second input windows (3 consecutive 30s epochs).

### Training parameters:
- **Epochs:** 100 (with early stopping, patience=50)
- **Batch size:** 100
- **Optimizer:** Adam (lr=1e-4)
- **Class balancing:** Oversampling minority classes
- **Cross-validation:** 20-fold leave-one-subject-out

### Quick test vs. full run:
- **Cell 3a**: Trains fold 0 only (~5-10 min) — use this to verify everything works
- **Cell 3b**: Trains all 20 folds (~2-4 hours total) — the full baseline experiment

In [ ]:
# ===== (Optional) Restore fold 0 model from Google Drive =====
# Run this if you previously saved fold 0 to Drive and want to skip retraining.
# Comment out if training from scratch.

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_MODEL = '/content/drive/MyDrive/SleepStageNet/models/fold0'
LOCAL_MODEL = 'output/model/v1/base/fold0'

if os.path.exists(DRIVE_MODEL):
    os.makedirs(os.path.dirname(LOCAL_MODEL), exist_ok=True)
    shutil.copytree(DRIVE_MODEL, LOCAL_MODEL, dirs_exist_ok=True)
    print(f"Restored fold 0 model from Drive!")
    !ls -la {LOCAL_MODEL}/deepfeaturenet/checkpoint/
else:
    print("No saved model found on Drive -- will train from scratch.")

In [ ]:
# ===== Cell 3b: Train all 20 folds (full baseline) =====
# Takes ~1 hour on T4 GPU. Skips folds with existing checkpoints.
# If Colab disconnects, just re-run this cell -- it picks up where it left off.

import os

OUTPUT_DIR = "output/model/v1/base"
N_FOLDS = 20

for fold_idx in range(N_FOLDS):
    ckpt_dir = os.path.join(OUTPUT_DIR, f"fold{fold_idx}", "deepfeaturenet", "checkpoint")
    if os.path.exists(ckpt_dir) and any(f.startswith(f"model_fold{fold_idx}") for f in os.listdir(ckpt_dir)):
        print(f"Fold {fold_idx}: checkpoint exists -- skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"Training fold {fold_idx}/{N_FOLDS - 1}")
    print(f"{'='*60}\n")

    !python train.py \
        --data_dir={DATA_DIR} \
        --output_dir={OUTPUT_DIR} \
        --n_folds={N_FOLDS} \
        --fold_idx={fold_idx} \
        --train_epochs=100 \
        --smooth_value=0 \
        --smooth_stats=False \
        --resume=False

print("\nAll folds complete!")

In [ ]:
# ===== Cell 3b: Train all 20 folds (full baseline) =====
# WARNING: This takes 2-4 hours. Consider running overnight.
# If Colab disconnects, set resume=True and re-run from the last completed fold.

import subprocess

OUTPUT_DIR = "output/model/v1/base"
N_FOLDS = 20
START_FOLD = 1   # Change this if resuming (skip already-completed folds)
END_FOLD = 19

for fold_idx in range(START_FOLD, END_FOLD + 1):
    print(f"\n{'='*60}")
    print(f"Training fold {fold_idx}/{END_FOLD}")
    print(f"{'='*60}\n")

    !python train.py \
        --data_dir={DATA_DIR} \
        --output_dir={OUTPUT_DIR} \
        --n_folds={N_FOLDS} \
        --fold_idx={fold_idx} \
        --train_epochs=100 \
        --smooth_value=0 \
        --smooth_stats=False \
        --resume=False

print("\nAll folds complete!")

## 4. Prediction and Evaluation

In [ ]:
# Run prediction on all trained folds
MODEL_DIR = "output/model/v1/base"
RESULTS_DIR = "output/results/v1/base"

!python predict.py \
    --data_dir={DATA_DIR} \
    --model_dir={MODEL_DIR} \
    --output_dir={RESULTS_DIR} \
    --cross_validation=True \
    --n_folds=20 \
    --MC_dropout=False \
    --smooth_stats=False

In [ ]:
# Generate summary metrics
!python summary_muquery.py --data_dir={RESULTS_DIR}

## 5. Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, cohen_kappa_score, classification_report
import glob

# Collect predictions from all folds
RESULTS_DIR = "output/results/v1/base"
result_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'output_fold*.npz')))

all_y_true = []
all_y_pred = []

for rf in result_files:
    with np.load(rf, allow_pickle=True) as f:
        # Flatten object arrays: each element is an array per subject
        all_y_true.extend(np.hstack(f['y_true']))
        all_y_pred.extend(np.hstack(f['y_pred']))

y_true = np.asarray(all_y_true, dtype=int)
y_pred = np.asarray(all_y_pred, dtype=int)

# Overall metrics
acc = np.mean(y_true == y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
weighted_f1 = f1_score(y_true, y_pred, average='weighted')
kappa = cohen_kappa_score(y_true, y_pred)

print(f"Overall Accuracy: {acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=['W', 'N1', 'N2', 'N3', 'REM']))

In [ ]:
# Confusion matrix
stage_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=stage_names, yticklabels=stage_names, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (counts)')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=stage_names, yticklabels=stage_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Confusion Matrix (normalized)')

plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved figures/confusion_matrices.png")

In [ ]:
# Per-class F1 scores and training curves
per_class_f1 = f1_score(y_true, y_pred, average=None)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Per-class F1 bar chart
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#9b59b6']
bars = axes[0].bar(stage_names, per_class_f1, color=colors)
for bar, f1_val in zip(bars, per_class_f1):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{f1_val:.3f}', ha='center', va='bottom', fontsize=11)
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Class F1 Score — DeepSleepNet-Lite Baseline')
axes[0].set_ylim(0, 1.05)
axes[0].axhline(y=macro_f1, color='gray', linestyle='--', label=f'Macro F1 = {macro_f1:.3f}')
axes[0].legend()

# Training curves (from fold 0 as example)
perf_file = 'output/model/v1/base/fold0/deepfeaturenet/perf_fold0.npz'
if os.path.exists(perf_file):
    with np.load(perf_file) as f:
        train_loss = f['train_loss']
        valid_loss = f['valid_loss']
        train_f1 = f['train_f1']
        valid_f1 = f['valid_f1']

    # Find actual number of epochs (non-zero)
    n_actual = np.max(np.nonzero(train_loss)) + 1
    epochs = range(1, n_actual + 1)

    axes[1].plot(epochs, train_loss[:n_actual], label='Train Loss', alpha=0.8)
    axes[1].plot(epochs, valid_loss[:n_actual], label='Valid Loss', alpha=0.8)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Training Curves — Fold 0')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'No training curves found\n(run training first)',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('figures/training_curves_and_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved figures/training_curves_and_f1.png")

In [ ]:
# Save results to CSV for the project report
import pandas as pd

results = {
    'Model': ['DeepSleepNet-Lite (baseline)'],
    'Accuracy': [f'{acc:.4f}'],
    'Macro_F1': [f'{macro_f1:.4f}'],
    'Weighted_F1': [f'{weighted_f1:.4f}'],
    'Kappa': [f'{kappa:.4f}'],
    'W_F1': [f'{per_class_f1[0]:.4f}'],
    'N1_F1': [f'{per_class_f1[1]:.4f}'],
    'N2_F1': [f'{per_class_f1[2]:.4f}'],
    'N3_F1': [f'{per_class_f1[3]:.4f}'],
    'REM_F1': [f'{per_class_f1[4]:.4f}'],
}
df = pd.DataFrame(results)
df.to_csv('results/baseline_results.csv', index=False)
print("Saved results/baseline_results.csv")
print(df.to_string(index=False))

## 6. Save to Google Drive

Save model checkpoints and results to Google Drive for persistence across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive
DRIVE_DIR = '/content/drive/MyDrive/SleepStageNet'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'data'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'models'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'results'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'figures'), exist_ok=True)

# Copy processed data (follow symlinks with -L)
!cp -rL {DATA_DIR} {DRIVE_DIR}/data/eeg_FpzCz_PzOz_v1
!cp data/data_split_v1.npz {DRIVE_DIR}/data/

# Copy model checkpoints
!cp -r output/model {DRIVE_DIR}/models/

# Copy results and figures
!cp -r output/results {DRIVE_DIR}/results/
!cp -r figures/* {DRIVE_DIR}/figures/
!cp results/baseline_results.csv {DRIVE_DIR}/results/

print(f"\nAll outputs saved to Google Drive: {DRIVE_DIR}")
!ls -la {DRIVE_DIR}